# Lektion 11 - Agent-zu-Agent (A2A) Protokoll


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Was ist das A2A-Protokoll?

Das **Agent-zu-Agent (A2A) Protokoll** ist ein offener Standard, der es KI-Agenten ermöglicht, zu kommunizieren,
sich gegenseitig zu entdecken und zusammenzuarbeiten — selbst wenn sie auf unterschiedlichen Frameworks aufgebaut oder
von verschiedenen Diensten gehostet werden.

Wichtige Konzepte:

- **Entdeckung** – Agenten veröffentlichen eine *Agent Card*, die ihre Fähigkeiten beschreibt, wodurch es
  anderen Agenten (oder Orchestratoren) erleichtert wird, den richtigen Spezialisten für eine Aufgabe zu finden.
- **Nachrichtenübermittlung** – Agenten tauschen strukturierte Nachrichten über ein gemeinsames Protokoll aus, so dass eine
  Anfrage von einem Agenten von einem anderen verstanden und erfüllt werden kann, unabhängig von dessen interner
  Implementierung.
- **Aufgabenlebenszyklus** – A2A definiert Zustände wie *submitted*, *working*, *completed*, und
  *failed*, wodurch der Orchestrator vollständige Sichtbarkeit darüber erhält, wie eine delegierte Aufgabe voranschreitet.

In dieser Lektion simulieren wir eine Zusammenarbeit im A2A-Stil, indem wir drei spezialisierte Reiseagenten
in einen Workflow einbinden, in dem jeder Agent seine Expertise einbringt und Ergebnisse an den nächsten weitergibt.


## Spezialisierte Reiseagenten erstellen


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Mehragenten-Zusammenarbeit über einen Workflow

Wir verbinden die drei Agenten zu einem sequentiellen Workflow, der die A2A-Nachrichtenübermittlung widerspiegelt:

1. **CurrencyExchangeAgent** erhält die Benutzeranfrage und liefert Währungsinformationen.
2. **ActivityPlannerAgent** erhält den angereicherten Kontext und fügt Aktivitätsempfehlungen hinzu.
3. **TravelManagerAgent** synthetisiert beide Eingaben zu einem abschließenden Reiseüberblick.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A im Produktionsbetrieb verstehen

In einer Produktionsumgebung eröffnet das A2A-Protokoll leistungsfähige dienstübergreifende Szenarien:

| Capability | Description |
|---|---|
| **Framework-übergreifende Interoperabilität** | Ein Agent, der mit einem Framework erstellt wurde, kann Aufgaben an einen Agenten delegieren, der mit jedem anderen A2A-konformen Framework erstellt wurde, wodurch echte organisationsübergreifende Interoperabilität ermöglicht wird. |
| **Service-Grenzen** | Agenten können in separaten Microservices, Cloud-Regionen oder sogar in verschiedenen Organisationen betrieben werden und dennoch nahtlos zusammenarbeiten. |
| **Dynamische Entdeckung** | Ein Orchestrator kann zur Laufzeit ein Agent Card-Register abfragen, um den am besten geeigneten Spezialisten für eine bestimmte Teilaufgabe zu finden. |
| **Streaming & Push-Benachrichtigungen** | A2A unterstützt Server-Sent Events (SSE) für Echtzeit-Fortschrittsupdates und Push-Benachrichtigungen für länger laufende Aufgaben. |

Der oben erstellte Workflow ist eine vereinfachte, prozessinterne Version dieses Musters. In einer echten Bereitstellung würde jeder Agent einen HTTP-Endpunkt bereitstellen, eine Agent Card veröffentlichen und über das A2A JSON-RPC-Protokoll kommunizieren.


## Zusammenfassung

In dieser Lektion haben Sie gelernt:

1. **Was das A2A-Protokoll ist** — ein offener Standard für Agent-zu-Agent-Entdeckung, Nachrichtenübermittlung und Aufgabenverwaltung.
2. **Wie man spezialisierte Agenten erstellt** — ein Currency Exchange-Agent, ein Activity Planner-Agent und ein Travel Manager-Orchestrator.
3. **Wie man Agenten in einen Workflow verkabelt** — Verwendung von `WorkflowBuilder`, um sequentielle Nachrichtenübermittlung zwischen Agenten zu modellieren.
4. **Wie A2A in der Produktion funktioniert** — ermöglicht frameworkübergreifende, dienstübergreifende Zusammenarbeit mit dynamischer Entdeckung und Streaming-Updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Haftungsausschluss:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner ursprünglichen Sprache ist als maßgebliche Quelle zu betrachten. Bei kritischen Informationen empfehlen wir eine professionelle menschliche Übersetzung. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
